# Ozon E-CUP — ночной конвейер абляций (полигон 2.17M)

Последовательно гонит матрицу экспериментов: модели одного класса × препроцессинг.
После каждого эксперимента результат дописывается в `/kaggle/working/ablation_results.json`
и печатается текущая сводка — обрыв сессии не теряет завершённые прогоны.

**Подключить**: датасет с данными (пути в конфиге) + файл полигона `llm_sample_2m.parquet`
(он в гите: data_polygon/) — проще всего докинуть его в тот же Kaggle-датасет.

Бюджет: ~10 часов на T4. Порядок экспериментов — по ценности (первые важнее).
Валидация у всех одна: общий llm-holdout (seed 13), confident, macro PR-AUC — сравнимо с таблицей README.


In [ ]:
import os, json, gc, time, math, random, re
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

BASE = "/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items"   # <-- поправь
ITEMS_PATH = f"{BASE}/items.parquet"
MATCHES_LLM_PATH = f"{BASE}/matches_llm.parquet"
POLYGON_PATH = f"{BASE}/llm_sample_2m.parquet"   # <-- докинь файл в датасет

RESULTS_PATH = "/kaggle/working/ablation_results.json"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda"
print(torch.cuda.get_device_name(0))

## Матрица экспериментов

Правь список под свои гипотезы. Поля: model, text (v1|v2), max_len, batch, accum, lr.
text_v1 — как в наших CE-прогонах (боевой формат tiny/stack). text_v2 — категория в тексте, атрибуты 460 симв.


In [ ]:
EXPERIMENTS = [
    # якорь: боевой рецепт tiny — связывает полигон с известными цифрами на 11M
    dict(name="tiny_v1_160",  model="cointegrated/rubert-tiny2",        text="v1", max_len=160, batch=256, accum=1, lr=2e-4),
    # главный вопрос: e5-small и польза препроцессинга v2 на нём
    dict(name="e5s_v1_160",   model="intfloat/multilingual-e5-small",   text="v1", max_len=160, batch=128, accum=2, lr=7e-5),
    dict(name="e5s_v2_192",   model="intfloat/multilingual-e5-small",   text="v2", max_len=192, batch=128, accum=2, lr=7e-5),
    # препроцессинг на tiny (дёшево, переносимо)
    dict(name="tiny_v2_192",  model="cointegrated/rubert-tiny2",        text="v2", max_len=192, batch=256, accum=1, lr=2e-4),
    dict(name="tiny_v1_224",  model="cointegrated/rubert-tiny2",        text="v1", max_len=224, batch=192, accum=1, lr=2e-4),
]

## Тексты товаров (оба варианта, только нужные товары)


In [ ]:
KEY_ORDER_V1 = ["бренд", "артикул", "партномер", "oem", "код", "модель", "размер",
                "цвет", "объем", "обьем", "вес", "тип", "материал", "количество"]

def build_text_v1(name, attributes, category, max_attr_chars=260):
    parts = [str(name) if name is not None else ""]
    try: attrs = json.loads(attributes) if isinstance(attributes, str) else {}
    except Exception: attrs = {}
    if isinstance(attrs, dict) and attrs:
        low = {str(k).lower(): str(v) for k, v in attrs.items() if v}
        picked, used = [], set()
        for want in KEY_ORDER_V1:
            for k, v in low.items():
                if want in k and k not in used:
                    picked.append(f"{k}:{v}"); used.add(k)
        rest = [f"{k}:{v}" for k, v in low.items() if k not in used]
        parts.append(" ; ".join(picked + rest)[:max_attr_chars])
    return " | ".join(parts)

def build_text_v2(name, attributes, category, max_attr_chars=460):
    base = build_text_v1(name, attributes, category, max_attr_chars)
    cat = str(category).lower() if category is not None else ""
    return f"категория: {cat} | {base}"

# сплиты: общий holdout из ПОЛНОГО matches_llm (seed 13), полигон — train
ml = pd.read_parquet(MATCHES_LLM_PATH)
parent = {}
def find(x):
    p = parent.setdefault(x, x)
    while p != parent[p]:
        parent[p] = parent[parent[p]]; p = parent[p]
    parent[x] = p; return p
for a, b in zip(ml.id1.values, ml.id2.values):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra
comp = np.fromiter((find(i) for i in ml.id1.values), dtype=np.int64, count=len(ml))
rng = np.random.RandomState(13)
uniq = np.unique(comp)
val_set = set(uniq[rng.rand(len(uniq)) < 0.03].tolist())
is_val = np.fromiter((c in val_set for c in comp), dtype=bool, count=len(ml))
holdout = ml[is_val].copy()
holdout = holdout[(holdout.target <= 0.2) | (holdout.target >= 0.8)]
holdout["target"] = (holdout.target >= 0.5).astype(np.int8)
del ml, parent, comp; gc.collect()

polygon = pd.read_parquet(POLYGON_PATH)
print(f"полигон: {len(polygon):,}; holdout: {len(holdout):,}")

need = set(polygon.id1) | set(polygon.id2) | set(holdout.id1) | set(holdout.id2)
raw = {}
cats = {}
f = pq.ParquetFile(ITEMS_PATH)
for b in f.iter_batches(columns=["id", "name", "attributes", "category"], batch_size=500_000):
    df = b.to_pandas()
    df = df[df["id"].isin(need)]
    for i, n, a, c in df.itertuples(index=False, name=None):
        raw[i] = (n, a); cats[i] = c
holdout["category"] = [cats[i] for i in holdout.id1]
print(f"товаров: {len(raw):,}")

TEXTS = {}
def get_texts(variant):
    if variant not in TEXTS:
        t0 = time.time()
        fn = build_text_v1 if variant == "v1" else build_text_v2
        TEXTS[variant] = {i: fn(n, a, cats[i]) for i, (n, a) in raw.items()}
        print(f"тексты {variant}: {time.time()-t0:.0f}s")
    return TEXTS[variant]

# fast-выборка для промежуточных замеров
holdout_fast = holdout.sample(min(60_000, len(holdout)), random_state=0)

## Обучение и оценка одного эксперимента


In [ ]:
class DS(Dataset):
    def __init__(self, df, texts):
        self.a = df.id1.values; self.b = df.id2.values
        self.y = df.target.values.astype(np.float32); self.t = texts
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.t[self.a[i]], self.t[self.b[i]], self.y[i]

def macro_ap(df, preds):
    z = df[["category", "target"]].copy(); z["p"] = preds
    return float(z.groupby("category").apply(
        lambda g: average_precision_score(g.target, g.p)).mean())

@torch.no_grad()
def predict(model, tok, df, texts, max_len, bs=512):
    model.eval()
    dl = DataLoader(DS(df, texts), batch_size=bs, num_workers=0, shuffle=False,
                    collate_fn=lambda b: tok([x[0] for x in b], [x[1] for x in b],
                        padding=True, truncation=True, max_length=max_len, return_tensors="pt"))
    out = []
    for enc in dl:
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast("cuda", torch.float16):
            out.append(torch.sigmoid(model(**enc).logits.squeeze(-1).float()).cpu().numpy())
    return np.concatenate(out)

def run_experiment(cfg):
    t_start = time.time()
    texts = get_texts(cfg["text"])
    tok = AutoTokenizer.from_pretrained(cfg["model"])
    model = AutoModelForSequenceClassification.from_pretrained(cfg["model"], num_labels=1).to(device)

    def collate(b):
        enc = tok([x[0] for x in b], [x[1] for x in b], padding=True,
                  truncation=True, max_length=cfg["max_len"], return_tensors="pt")
        return enc, torch.tensor([x[2] for x in b])

    dl = DataLoader(DS(polygon, texts), batch_size=cfg["batch"], shuffle=True,
                    num_workers=0, drop_last=True, collate_fn=collate)
    steps = len(dl) // cfg["accum"]
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, steps // 30, steps)
    scaler = torch.amp.GradScaler()
    lossf = nn.BCEWithLogitsLoss()
    model.train()
    t0, seen = time.time(), 0
    for bi, (enc, y) in enumerate(dl):
        enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}
        y = y.to(device, non_blocking=True)
        with torch.autocast("cuda", torch.float16):
            loss = lossf(model(**enc).logits.squeeze(-1), y) / cfg["accum"]
        scaler.scale(loss).backward()
        if (bi + 1) % cfg["accum"] == 0:
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        seen += len(y)
        if seen % (cfg["batch"] * cfg["accum"] * 500) < cfg["batch"]:
            print(f"  [{cfg['name']}] {seen:,}/{len(polygon):,} {seen/(time.time()-t0):.0f} pair/s", flush=True)
    fast = macro_ap(holdout_fast, predict(model, tok, holdout_fast, texts, cfg["max_len"]))
    full = macro_ap(holdout, predict(model, tok, holdout, texts, cfg["max_len"]))
    res = dict(cfg, fast_macro=round(fast, 4), full_macro=round(full, 4),
               minutes=round((time.time()-t_start)/60, 1))
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

## Конвейер: гоним всё, результаты пишутся после каждого


In [ ]:
results = []
if os.path.exists(RESULTS_PATH):
    results = json.load(open(RESULTS_PATH))
    print("продолжаю, уже готово:", [r["name"] for r in results])

done = {r["name"] for r in results}
for cfg in EXPERIMENTS:
    if cfg["name"] in done:
        continue
    print(f"=== {cfg['name']} ===", flush=True)
    res = run_experiment(cfg)
    results.append(res)
    json.dump(results, open(RESULTS_PATH, "w"), ensure_ascii=False, indent=1)
    print(pd.DataFrame(results)[["name","model","text","max_len","fast_macro","full_macro","minutes"]]
          .to_string(index=False), flush=True)

print("ГОТОВО")
summary = pd.DataFrame(results).sort_values("full_macro", ascending=False)
print(summary[["name","model","text","max_len","fast_macro","full_macro","minutes"]].to_string(index=False))

## Как читать утром

- `full_macro` — главная колонка (общий llm-holdout, сравнимо с README, но уровень ниже полных данных на ~0.02-0.04).
- Сравнивай ПАРАМИ: e5s_v1 vs e5s_v2 → польза препроцессинга; tiny_v1_160 vs tiny_v1_224 → польза длины; e5s vs tiny — НЕ сравнивать (разные классы, e5 недотренирован на 2M).
- Победителей — на полные данные (11M), и только их цифры идут в таблицу README как кандидаты.
